In [ ]:
!uv pip install -q \
    sentence-transformers \
    rank_bm25 \
    faiss \
    pyarrow \
    pandas \
    numpy \
    tqdm \
    einops \
    transformers \
    accelerate

print("✅ All dependencies installed.")




In [5]:
!pip install --upgrade --force-reinstall sympy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 34.6 MB/s eta 0:00:00
  Attempting uninstall: mpmath
    Found existing installation: mpmath 1.3.0
    Uninstalling mpmath-1.3.0:
      Successfully uninstalled mpmath-1.3.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports & Config
# ─────────────────────────────────────────────────────────────────────────────
import gzip
import json
import math
import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date, datetime
from typing import Optional
from tqdm.auto import tqdm

import torch
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss

# ── Config ────────────────────────────────────────────────────────────────────
EMBED_MODEL_NAME  = "microsoft/harrier-oss-v1-0.6b"     # 1024-dim, large context window
EMBED_BATCH_SIZE  = 100                                 # Optimized to fit 15GB VRAM (T4) at 4096 seq len
MAX_TOKEN_WINDOW  = 4096                                # Accuracy over speed with large context
EMBED_DIM         = 1024

# Set Local Base Directory
import os
BASE_DIR = r'c:\Home\Projects\Mainstream\Chronos\India_runs_data_and_ai_challenge'
os.makedirs(BASE_DIR, exist_ok=True)
INPUT_FILE = os.path.join(BASE_DIR, "candidates.jsonl")
EMBEDDINGS_FILE = os.path.join(BASE_DIR, "embeddings.npy")

from pathlib import Path
ARTIFACTS_DIR = Path(BASE_DIR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Running on: {DEVICE.upper()}")
print(f"   Model:      {EMBED_MODEL_NAME}")
print(f"   Batch size: {EMBED_BATCH_SIZE}")

C:\Users\VIVEK\miniconda3\envs\andrea\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Running on: CUDA
   Model:      microsoft/harrier-oss-v1-0.6b
   Batch size: 100


In [2]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: NVIDIA GeForce RTX 4050 Laptop GPU


In [11]:
!uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126


Using Python 3.13.13 environment at: C:\Users\VIVEK\anaconda3\envs\redrob
Resolved 14 packages in 5.08s
Prepared 3 packages in 2.37s
Installed 3 packages in 127ms
 + pillow==12.2.0
 + torchaudio==2.11.0+cu126
 + torchvision==0.27.0+cu126


In [3]:
import os

# Check if candidates.jsonl or candidates.jsonl.gz exists in BASE_DIR
if os.path.exists(INPUT_FILE):
    CANDIDATES_PATH = INPUT_FILE
    print(f"✅ Found input file: {CANDIDATES_PATH}")
elif os.path.exists(INPUT_FILE + ".gz"):
    CANDIDATES_PATH = INPUT_FILE + ".gz"
    print(f"✅ Found compressed input file: {CANDIDATES_PATH}")
else:
    print(f"❌ Input file not found at {INPUT_FILE} or {INPUT_FILE}.gz")
    raise FileNotFoundError(f"Please place candidates.jsonl in {BASE_DIR}")

✅ Found input file: c:\Home\Projects\Mainstream\Chronos\India_runs_data_and_ai_challenge\candidates.jsonl


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Load & Parse Candidates (lazy streaming, memory-efficient)
# ─────────────────────────────────────────────────────────────────────────────

# ── Rubric constants (mirrored from feature_engineering.py) ──────────────────
SERVICES_FIRMS_BLACKLIST = {
    "tcs", "tata consultancy services",
    "infosys", "infosys technologies",
    "wipro", "wipro technologies",
    "accenture",
    "cognizant", "cognizant technology solutions",
    "capgemini",
    "hcl", "hcl technologies",
    "tech mahindra",
    "mphasis", "hexaware",
    "ltimindtree", "l&t infotech", "larsen toubro infotech",
    "virtusa", "niit technologies", "sasken",
}

MUST_HAVE_SKILLS = {
    "embeddings": [
        "sentence-transformers", "sentence_transformers", "sbert",
        "bge", "e5", "openai embeddings", "ada", "text-embedding",
        "dense retrieval", "bi-encoder", "huggingface embeddings",
    ],
    "vector_databases": [
        "pinecone", "weaviate", "qdrant", "milvus", "faiss",
        "opensearch", "elasticsearch", "chromadb", "pgvector",
        "redis vector", "annoy", "nmslib", "scann",
    ],
    "python": ["python"],
    "ranking_evaluation": [
        "ndcg", "mrr", "map", "mean average precision",
        "precision at k", "ranking evaluation", "a/b testing",
        "learning to rank", "ltr", "offline evaluation",
        "recsys evaluation", "online evaluation",
    ],
}

BONUS_SKILLS = {
    "llm_finetuning":    ["lora", "qlora", "peft", "fine-tuning", "finetuning", "sft", "rlhf"],
    "ltr_models":        ["lambdamart", "ranknet", "listwise", "pairwise ranking",
                          "xgboost rank", "lightgbm rank"],
    "hrtech":            ["hr tech", "hrtech", "recruiting tech", "talent platform",
                          "ats", "job matching"],
    "distributed_infra": ["spark", "kafka", "ray", "kubernetes", "triton",
                          "torchserve", "distributed inference", "dask"],
}

PRODUCTION_KEYWORDS = [
    "deployed", "production", "shipped", "launched", "scaled",
    "a/b test", "millions", "users", "real-time", "latency",
    "inference", "ranking system", "recommendation", "search engine",
    "retrieval", "indexing", "pipeline at scale", "live system",
    "serving", "online", "throughput",
]

SHIP_VERBS = [
    "shipped", "launched", "built", "delivered", "deployed",
    "improved", "reduced", "increased", "scaled", "productionized",
    "owned", "led", "drove", "created", "redesigned", "rewrote",
]

PREFERRED_LOCATIONS = {
    "pune": 1.00, "noida": 1.00,
    "hyderabad": 0.90, "mumbai": 0.90,
    "delhi ncr": 0.90, "delhi": 0.90,
    "gurgaon": 0.90, "gurugram": 0.90,
    "bangalore": 0.85, "bengaluru": 0.85,
    "chennai": 0.80,
    "kolkata": 0.70, "ahmedabad": 0.65,
}

PROFICIENCY_WEIGHTS = {
    "beginner": 0.20, "intermediate": 0.50,
    "advanced": 0.80, "expert": 1.00,
}

TODAY = date.today()

# ── Load ──────────────────────────────────────────────────────────────────────
def load_candidates(path: str) -> list[dict]:
    opener = gzip.open if path.endswith(".gz") else open
    candidates = []
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                candidates.append(json.loads(line))
    return candidates

candidates = load_candidates(CANDIDATES_PATH)

from datetime import datetime
def days_since(date_str):
    if not date_str:
        return 730.0
    try:
        d = datetime.strptime(date_str, "%Y-%m-%d").date()
        return max(0.0, (TODAY - d).days)
    except:
        return 730.0
def is_services_firm(company):
    c = (company or "").lower().strip()
    return any(firm in c for firm in SERVICES_FIRMS_BLACKLIST)
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1 — Hard Binary Masks (applied to corpus before feature extraction)
# Candidates that fail any mask are dropped entirely from all downstream steps.
# ─────────────────────────────────────────────────────────────────────────────

_n_before = len(candidates)

ROLE_DISQUALIFY_TERMS = {"marketing", "sales", "hr", "recruiter", "talent"}

def _passes_masks(c: dict) -> bool:
    profile = c.get("profile") or {}
    redrob  = c.get("redrob_signals") or {}
    career  = c.get("career_history") or []

    # ── Mask 1: Inactivity ───────────────────────────────────────────────────
    # Drop if last_active_date is > 90 days from today.
    if days_since(redrob.get("last_active_date")) > 90:
        return False

    # ── Mask 2: Unresponsiveness ─────────────────────────────────────────────
    # Drop if recruiter_response_rate < 0.15 AND on platform > 30 days.
    rr = float(redrob.get("recruiter_response_rate") or 0.0)
    joined_days = days_since(redrob.get("signup_date"))
    if rr < 0.15 and joined_days > 30:
        return False

    # ── Mask 3: Role Disqualification ────────────────────────────────────────
    # Drop if current_title contains any disqualifying role keyword.
    cur_title = (profile.get("current_title") or "").lower()
    if any(term in cur_title for term in ROLE_DISQUALIFY_TERMS):
        return False

    # ── Mask 4: Consulting Disqualification ──────────────────────────────────
    # Drop if 100% of career_history companies are in SERVICES_FIRMS_BLACKLIST.
    if career and all(is_services_firm(r.get("company") or "") for r in career):
        return False

    return True

candidates = [c for c in candidates if _passes_masks(c)]

print(f"\n🔒 Hard Binary Masks applied:")
print(f"   Corpus before masks : {_n_before:,}")
print(f"   Corpus after  masks : {len(candidates):,}")
print(f"   Dropped             : {_n_before - len(candidates):,} ({(_n_before - len(candidates)) / _n_before * 100:.1f}%)")
print(f"✅ Loaded {len(candidates):,} candidates")
print(f"   Sample keys: {list(candidates[0].keys())}")


🔒 Hard Binary Masks applied:
   Corpus before masks : 100,000
   Corpus after  masks : 19,607
   Dropped             : 80,393 (80.4%)
✅ Loaded 19,607 candidates
   Sample keys: ['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals']


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Build Rich Profile Text for Embedding
# Uses a 2048-token window to prevent truncation of long resumes.
# nomic-embed-text-v1.5 requires the task prefix for retrieval.
# ─────────────────────────────────────────────────────────────────────────────

def build_profile_text(c: dict) -> str:
    """
    Concatenate all meaningful text fields into a single document for embedding.
    Ordered by semantic importance for the JD (skills first, then experience).
    Prefixed with 'search_document:' as required by nomic-embed-text-v1.5.
    """
    profile = c.get("profile") or {}
    skills  = c.get("skills") or []
    career  = c.get("career_history") or []

    parts = []

    # Identity
    title   = profile.get("current_title", "")
    company = profile.get("current_company", "")
    yoe     = profile.get("years_of_experience", "")
    if title:   parts.append(f"Title: {title}")
    if company: parts.append(f"Company: {company}")
    if yoe:     parts.append(f"Experience: {yoe} years")

    # Skills — sorted by proficiency descending for better semantic signal
    sorted_skills = sorted(
        skills,
        key=lambda s: PROFICIENCY_WEIGHTS.get(s.get("proficiency", "beginner"), 0),
        reverse=True
    )
    skill_strs = []
    for s in sorted_skills:
        name  = s.get("name", "")
        prof  = s.get("proficiency", "")
        dur   = s.get("duration_months", 0)
        if name:
            skill_strs.append(f"{name} ({prof}, {dur}mo)")
    if skill_strs:
        parts.append("Skills: " + "; ".join(skill_strs))

    # Career history — concatenate descriptions
    for role in career[:6]:  # cap at 6 roles to stay within token window
        role_title   = role.get("title", "")
        role_company = role.get("company", "")
        role_desc    = role.get("description", "")
        dur          = role.get("duration_months", 0)
        if role_title or role_desc:
            parts.append(
                f"Role: {role_title} @ {role_company} ({dur}mo). {role_desc}"
            )

    # Education
    education = c.get("education") or []
    for edu in education[:2]:
        degree = edu.get("degree", "")
        field  = edu.get("field_of_study", "")
        school = edu.get("institution", "")
        if degree or school:
            parts.append(f"Education: {degree} in {field} from {school}")

    raw_text = "\n".join(parts)

    # nomic-embed-text-v1.5 requires task prefix for asymmetric retrieval
    return f"search_document: {raw_text}"


# Build JD query text (will be used for retrieval at runtime by Rust engine)
JD_QUERY_TEXT = (
    "search_query: Senior AI Engineer with 5-9 years experience in production ML systems. "
    "Must have: dense embeddings (sentence-transformers, BGE, E5, FAISS), vector databases "
    "(Pinecone, Weaviate, Qdrant, FAISS), strong Python, ranking evaluation (NDCG, MRR, LTR). "
    "Nice-to-have: LLM fine-tuning (LoRA, QLoRA), LambdaMART, distributed infra (Spark, Ray, Kubernetes). "
    "Product-company background required. No pure IT services careers."
)

# Build corpus texts
print("Building profile texts ...")
candidate_ids   = [c["candidate_id"] for c in candidates]
profile_texts   = [build_profile_text(c) for c in tqdm(candidates, desc="Texts")]

print(f"✅ Built {len(profile_texts):,} profile texts")
print(f"   Sample (first 300 chars):\n   {profile_texts[0]}...")


Building profile texts ...


Texts: 100%|██████████| 19607/19607 [00:00<00:00, 53383.06it/s]

✅ Built 19,607 profile texts
   Sample (first 300 chars):
   search_document: Title: Backend Engineer
Company: Mindtree
Experience: 6.9 years
Skills: NLP (advanced, 26mo); Image Classification (advanced, 40mo); Fine-tuning LLMs (advanced, 36mo); Speech Recognition (advanced, 33mo); TTS (advanced, 60mo); Milvus (advanced, 35mo); GANs (advanced, 19mo); Tailwind (intermediate, 13mo); Weights & Biases (intermediate, 30mo); Photoshop (intermediate, 24mo); LoRA (intermediate, 28mo); Apache Beam (intermediate, 9mo); BentoML (intermediate, 36mo); Statistical Modeling (intermediate, 8mo); AWS (beginner, 8mo); Flask (beginner, 15mo); GCP (beginner, 2mo)
Role: Backend Engineer @ Mindtree (27mo). Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make su

In [8]:
profile_texts[0]
import glob
import os
from pathlib import Path

# Clear stale embedding checkpoints
stale_files = glob.glob(os.path.join(ARTIFACTS_DIR, "embeddings*.npy"))
for f in stale_files:
    os.remove(f)
    print(f"🗑️ Deleted stale artifact: {os.path.basename(f)}")

🗑️ Deleted stale artifact: embeddings.npy
🗑️ Deleted stale artifact: embeddings_chunk_0.npy
🗑️ Deleted stale artifact: embeddings_chunk_10000.npy
🗑️ Deleted stale artifact: embeddings_chunk_12000.npy
🗑️ Deleted stale artifact: embeddings_chunk_14000.npy
🗑️ Deleted stale artifact: embeddings_chunk_16000.npy
🗑️ Deleted stale artifact: embeddings_chunk_18000.npy
🗑️ Deleted stale artifact: embeddings_chunk_2000.npy
🗑️ Deleted stale artifact: embeddings_chunk_4000.npy
🗑️ Deleted stale artifact: embeddings_chunk_6000.npy
🗑️ Deleted stale artifact: embeddings_chunk_8000.npy


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Dense Embeddings via nomic-embed-text-v1.5
# ~10-15 minutes for 10k candidates on T4 GPU
# Output: embeddings.npy  (shape: [N, 768], dtype: float16)
# ─────────────────────────────────────────────────────────────────────────────

print(f"Loading {EMBED_MODEL_NAME} ...")
model = SentenceTransformer(
    EMBED_MODEL_NAME,
    trust_remote_code=True,   # required for nomic-embed-text-v1.5
    device=DEVICE,
)
print(f"✅ Model loaded. Max token window: {model.max_seq_length}")

# Override max seq length to use full 2048 window
model.max_seq_length = MAX_TOKEN_WINDOW

# ── Batch embed with Checkpointing ─────────────────────────────────────────────
print(f"\nGenerating embeddings in chunks to allow resuming...")
CHUNK_SIZE = 2000  # Save a checkpoint every 2000 candidates
all_embeddings = []
start_idx = 0
EMB_PATH = Path(EMBEDDINGS_FILE)

# ── PRE-PROCESSING STEP ──
# Store original order to map back later
print("\nSorting corpus by length to minimize padding overhead...")
indexed_texts = sorted(enumerate(profile_texts), key=lambda x: len(x[1]))
sorted_texts = [x[1] for x in indexed_texts]
sorted_indices = [x[0] for x in indexed_texts]

# 1) Try to resume from checkpoints
import glob
import gc
checkpoint_files = sorted(ARTIFACTS_DIR.glob("embeddings_chunk_*.npy"), 
                          key=lambda x: int(x.stem.split('_')[-1].split('.')[0]))

if checkpoint_files:
    print(f"🔄 Found {len(checkpoint_files)} checkpoints. Resuming...")
    for ckpt in checkpoint_files:
        chunk_emb = np.load(ckpt)
        all_embeddings.append(chunk_emb)
        start_idx += chunk_emb.shape[0]
    print(f"   Resumed {start_idx} embeddings.")
    
    # Pre-populate progressive embeddings file with resumed data
    if all_embeddings:
        current_stacked = np.vstack(all_embeddings)
        progressive_embeddings = np.zeros((len(sorted_texts), current_stacked.shape[1]), dtype=np.float16)
        computed_indices = sorted_indices[:current_stacked.shape[0]]
        progressive_embeddings[computed_indices] = current_stacked
        np.save(EMB_PATH, progressive_embeddings)
        print(f"   💾 Initialized progressive embeddings → {EMB_PATH.name}")

# 2) Process remaining texts
if start_idx < len(sorted_texts):
    for i in range(start_idx, len(sorted_texts), CHUNK_SIZE):
        chunk_texts = sorted_texts[i : i + CHUNK_SIZE]
        print(f"\nEmbedding chunk {i} to {i + len(chunk_texts)} ...")
        
        # We use a batch size that fits the 15GB T4 GPU VRAM
        chunk_embeddings_f32 = model.encode(
            chunk_texts,
            batch_size=EMBED_BATCH_SIZE,
            show_progress_bar=True,
            normalize_embeddings=True,       # L2-norm → cosine sim = dot product
            convert_to_numpy=True,
        )
        
        # Downcast to FP16 to keep footprint < 5 GB as per blueprint
        chunk_embeddings_f16 = chunk_embeddings_f32.astype(np.float16)
        all_embeddings.append(chunk_embeddings_f16)
        
        # Force Memory Cleanup
        del chunk_embeddings_f32
        gc.collect()
        torch.cuda.empty_cache()
        
        # Save checkpoint to Google Drive / Artifacts dir
        chunk_path = ARTIFACTS_DIR / f"embeddings_chunk_{i}.npy"
        np.save(chunk_path, chunk_embeddings_f16)
        print(f"   💾 Saved chunk checkpoint → {chunk_path.name}")
        
        # Save running state of the full embeddings file to prevent loss of partial runs
        # Restoring original index positions so the file is immediately usable mapping to candidate_ids
        current_stacked = np.vstack(all_embeddings)
        progressive_embeddings = np.zeros((len(sorted_texts), current_stacked.shape[1]), dtype=np.float16)
        computed_indices = sorted_indices[:current_stacked.shape[0]]
        progressive_embeddings[computed_indices] = current_stacked
        np.save(EMB_PATH, progressive_embeddings)
        print(f"   💾 Updated progressive full embeddings → {EMB_PATH.name} ({current_stacked.shape[0]}/{len(sorted_texts)} done)")

embeddings_f16_sorted = np.vstack(all_embeddings)

# Restore original indexing order for the final array
embeddings_f16 = np.empty_like(embeddings_f16_sorted)
embeddings_f16[sorted_indices] = embeddings_f16_sorted

print(f"\n✅ All embeddings generated")
print(f"   Shape:  {embeddings_f16.shape}")
print(f"   Dtype:  {embeddings_f16.dtype}")
print(f"   Memory: {embeddings_f16.nbytes / 1024**2:.1f} MB")

# Save final
np.save(EMB_PATH, embeddings_f16)
print(f"\n💾 Saved Final → {EMB_PATH}")

# Also save candidate ID order (critical for runtime lookup)
ID_PATH = ARTIFACTS_DIR / "candidate_ids.json"
with open(ID_PATH, "w") as f:
    json.dump(candidate_ids, f)
print(f"💾 Saved → {ID_PATH}")

Loading microsoft/harrier-oss-v1-0.6b ...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6100.55it/s]


✅ Model loaded. Max token window: 32768

Generating embeddings in chunks to allow resuming...

Sorting corpus by length to minimize padding overhead...

Embedding chunk 0 to 2000 ...


Batches: 100%|██████████| 20/20 [00:47<00:00,  2.40s/it]


   💾 Saved chunk checkpoint → embeddings_chunk_0.npy
   💾 Updated progressive full embeddings → embeddings.npy (2000/19607 done)

Embedding chunk 2000 to 4000 ...


...

   💾 Saved chunk checkpoint → embeddings_chunk_14000.npy
   💾 Updated progressive full embeddings → embeddings.npy (16000/19607 done)

Embedding chunk 16000 to 18000 ...


Batches: 100%|██████████| 20/20 [02:18<00:00,  6.91s/it]


   💾 Saved chunk checkpoint → embeddings_chunk_16000.npy
   💾 Updated progressive full embeddings → embeddings.npy (18000/19607 done)

Embedding chunk 18000 to 19607 ...


Batches: 100%|██████████| 17/17 [01:53<00:00,  6.67s/it]


   💾 Saved chunk checkpoint → embeddings_chunk_18000.npy
   💾 Updated progressive full embeddings → embeddings.npy (19607/19607 done)

✅ All embeddings generated
   Shape:  (19607, 1024)
   Dtype:  float16
   Memory: 38.3 MB

💾 Saved Final → c:\Home\Projects\Mainstream\Chronos\India_runs_data_and_ai_challenge\embeddings.npy
💾 Saved → c:\Home\Projects\Mainstream\Chronos\India_runs_data_and_ai_challenge\candidate_ids.json


In [23]:
import torch
import gc

   

# 2. Invoke the Python garbage collector
gc.collect()

# 3. Empty the PyTorch caching allocator
torch.cuda.empty_cache()

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Build FAISS Index (IVFFlat for fast ANN search)
# Output: faiss_index.bin
# ─────────────────────────────────────────────────────────────────────────────

print("Building FAISS index ...")

# Use FP32 for FAISS (FP16 not natively supported)
embeddings_f32_faiss = embeddings_f16.astype(np.float32)
n_candidates = embeddings_f32_faiss.shape[0]

# IVFFlat: good balance of speed vs accuracy for 10k-100k scale
# nlist = sqrt(N) is a good heuristic
nlist = max(1, int(math.sqrt(n_candidates)))
print(f"   N={n_candidates:,}, nlist={nlist}")

quantizer = faiss.IndexFlatIP(EMBED_DIM)   # inner product (works with L2-normed vecs)

# Check if both DEVICE is cuda and faiss has GPU support compiled
if DEVICE == "cuda" and hasattr(faiss, "StandardGpuResources"):
    # Build on GPU for speed
    res = faiss.StandardGpuResources()
    gpu_index = faiss.GpuIndexIVFFlat(res, EMBED_DIM, nlist, faiss.METRIC_INNER_PRODUCT)
    gpu_index.train(embeddings_f32_faiss)
    gpu_index.add(embeddings_f32_faiss)
    # Move back to CPU for serialization
    index = faiss.index_gpu_to_cpu(gpu_index)
else:
    # CPU fallback (very fast anyway for N=100,000)
    print("   Building on CPU...")
    index = faiss.IndexIVFFlat(quantizer, EMBED_DIM, nlist, faiss.METRIC_INNER_PRODUCT)
    index.train(embeddings_f32_faiss)
    index.add(embeddings_f32_faiss)


print(f"✅ FAISS index built. Total vectors: {index.ntotal:,}")

FAISS_PATH = ARTIFACTS_DIR / "faiss_index.bin"
faiss.write_index(index, str(FAISS_PATH))
print(f"💾 Saved → {FAISS_PATH}")

Building FAISS index ...
   N=19,607, nlist=140
   Building on CPU...
✅ FAISS index built. Total vectors: 19,607
💾 Saved → c:\Home\Projects\Mainstream\Chronos\India_runs_data_and_ai_challenge\faiss_index.bin


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Sparse Lexical Index (BM25)
# Anchors exact technical noun matches: Pinecone, RAG, NDCG, etc.
# Output: bm25_index.pkl
# ─────────────────────────────────────────────────────────────────────────────

def tokenize_for_bm25(text: str) -> list[str]:
    """
    Lowercase, remove punctuation, split on whitespace.
    Keeps technical terms intact (e.g., 'bert', 'faiss', 'ndcg').
    """
    text = text.lower()
    text = re.sub(r"[^\w\s\-]", " ", text)
    tokens = text.split()
    # Keep tokens >= 2 chars to remove noise
    return [t for t in tokens if len(t) >= 2]

# Strip the nomic prefix for BM25 (we only want the raw text)
raw_texts = [t.replace("search_document: ", "") for t in profile_texts]

print("Tokenizing for BM25 ...")
tokenized_corpus = [tokenize_for_bm25(t) for t in tqdm(raw_texts, desc="Tokenizing")]

print("Building BM25 index ...")
bm25 = BM25Okapi(tokenized_corpus)

BM25_PATH = ARTIFACTS_DIR / "bm25_index.pkl"
with open(BM25_PATH, "wb") as f:
    pickle.dump({"bm25": bm25, "candidate_ids": candidate_ids}, f)

print(f"✅ BM25 index built over {len(tokenized_corpus):,} documents")
print(f"   Avg tokens per doc: {sum(len(t) for t in tokenized_corpus) / len(tokenized_corpus):.0f}")
print(f"💾 Saved → {BM25_PATH}")

Tokenizing for BM25 ...


Tokenizing: 100%|██████████| 19607/19607 [00:01<00:00, 11279.78it/s]


Building BM25 index ...
✅ BM25 index built over 19,607 documents
   Avg tokens per doc: 231
💾 Saved → c:\Home\Projects\Mainstream\Chronos\India_runs_data_and_ai_challenge\bm25_index.pkl


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Tabular Feature Matrix
# Mirrors feature_engineering.py rubric logic exactly.
# Adds exponential decay features and boolean JD requirement flags.
# Output: features.parquet
# ─────────────────────────────────────────────────────────────────────────────

# ── Helpers ───────────────────────────────────────────────────────────────────

def clamp(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, x))

def log_scale(v, cap=50):
    return clamp(math.log1p(max(0, v)) / math.log1p(cap))

def normalize_minmax(v, lo, hi):
    return clamp((v - lo) / (hi - lo + 1e-9))

def days_since(date_str):
    if not date_str:
        return 730.0
    try:
        d = datetime.strptime(date_str, "%Y-%m-%d").date()
        return max(0.0, (TODAY - d).days)
    except:
        return 730.0

def is_services_firm(company):
    c = (company or "").lower().strip()
    return any(firm in c for firm in SERVICES_FIRMS_BLACKLIST)

def skill_hit(skill_name, keywords):
    sn = (skill_name or "").lower()
    return any(kw in sn for kw in keywords)

# ── Feature builder ───────────────────────────────────────────────────────────

def build_features(c: dict) -> dict:
    profile = c.get("profile") or {}
    career  = c.get("career_history") or []
    skills  = c.get("skills") or []
    redrob  = c.get("redrob_signals") or {}

    yoe        = float(profile.get("years_of_experience") or 0)
    skill_names = [(s.get("name") or "").lower() for s in skills]
    cur_title   = (profile.get("current_title") or "").lower()

    # ── DIM 1: Product Experience ─────────────────────────────────────────────
    total_months   = sum(int(r.get("duration_months") or 0) for r in career)
    product_months = sum(
        int(r.get("duration_months") or 0)
        for r in career if not is_services_firm(r.get("company") or "")
    )
    product_ratio  = product_months / max(total_months, 1)

    prod_evidence = sum(
        1 for r in career
        if sum(1 for kw in PRODUCTION_KEYWORDS if kw in (r.get("description") or "").lower()) >= 2
    )

    tenures = [int(r.get("duration_months") or 0) for r in career]
    short_stints = sum(1 for t in tenures if t < 18)

    if   product_ratio == 0.0: product_score = 0.05
    elif product_ratio >= 0.80: product_score = 1.00
    elif product_ratio >= 0.60: product_score = 0.80
    elif product_ratio >= 0.40: product_score = 0.60
    else:                       product_score = 0.30

    dim1 = (
        product_score              * 0.60 +
        clamp(prod_evidence / 2.0) * 0.25 +
        clamp(1.0 - short_stints * 0.25) * 0.15
    )

    # ── DIM 2: Modern AI Depth ────────────────────────────────────────────────
    all_kws = {kw for kws in {**MUST_HAVE_SKILLS, **BONUS_SKILLS}.values() for kw in kws}
    skill_lookup = {(s.get("name") or "").lower(): s for s in skills}

    groups_hit = sum(
        1 for grp_kws in MUST_HAVE_SKILLS.values()
        if any(skill_hit(sn, grp_kws) for sn in skill_lookup)
    )
    must_have_score = groups_hit / len(MUST_HAVE_SKILLS)

    ai_scores = []
    for sn, s in skill_lookup.items():
        if not any(kw in sn for kw in all_kws):
            continue
        p = PROFICIENCY_WEIGHTS.get(s.get("proficiency"), 0.20)
        e = log_scale(s.get("endorsements") or 0, cap=50)
        d = clamp((s.get("duration_months") or 0) / 12.0)
        ai_scores.append(p * 0.50 + e * 0.20 + d * 0.20)

    if ai_scores:
        top3 = sorted(ai_scores, reverse=True)[:3]
        quality_score = sum(top3) / 3 * 0.70 + sum(ai_scores) / len(ai_scores) * 0.30
    else:
        quality_score = 0.0

    bonus_hit = sum(
        1 for grp_kws in BONUS_SKILLS.values()
        if any(skill_hit(sn, grp_kws) for sn in skill_lookup)
    )
    github_raw  = redrob.get("github_activity_score", -1)
    github_norm = normalize_minmax(max(0, github_raw), 0, 100) if github_raw >= 0 else 0.30
    bonus_score = clamp((bonus_hit / len(BONUS_SKILLS)) * 0.50 + github_norm * 0.50)

    dim2 = must_have_score * 0.50 + quality_score * 0.30 + bonus_score * 0.20

    # ── DIM 3: Operational Scrappiness ────────────────────────────────────────
    icr = clamp(redrob.get("interview_completion_rate") or 0.5)
    ship_score = icr * 0.70 + github_norm * 0.30

    verb_counts = [
        sum(1 for v in SHIP_VERBS if v in (r.get("description") or "").lower())
        for r in career if (r.get("description") or "")
    ]
    desc_score = clamp(sum(verb_counts) / len(verb_counts) / 5.0) if verb_counts else 0.0

    saved_score  = log_scale(redrob.get("saved_by_recruiters_30d") or 0, cap=10)
    apps_score   = clamp((redrob.get("applications_submitted_30d") or 0) / 5.0)
    market_score = saved_score * 0.60 + apps_score * 0.40

    dim3 = ship_score * 0.50 + desc_score * 0.30 + market_score * 0.20

    # ── DIM 4: Actual Availability ────────────────────────────────────────────
    dsince = days_since(redrob.get("last_active_date"))
    # Exponential decay feature as per blueprint
    recency_exp   = math.exp(-0.05 * dsince / 30)   # exp decay over months
    recency_log   = 1.0 / (1.0 + math.log1p(dsince))

    rr    = clamp(redrob.get("recruiter_response_rate") or 0.50)
    hours = float(redrob.get("avg_response_time_hours") or 24)
    t_score = 1.0 if hours <= 4 else (0.8 if hours <= 24 else (0.5 if hours <= 48 else 0.2))
    resp_score = rr * 0.70 + t_score * 0.30

    # Response rate with exponential decay as per blueprint
    response_rate_decayed = rr * math.exp(-0.05 * dsince / 30)

    otw         = 1.0 if redrob.get("open_to_work_flag") else 0.40
    verified    = (
        (1 if redrob.get("verified_email")    else 0) +
        (1 if redrob.get("verified_phone")    else 0) +
        (1 if redrob.get("linkedin_connected") else 0)
    ) / 3.0
    completeness = clamp((redrob.get("profile_completeness_score") or 50) / 100.0)
    mkt_avail    = otw * 0.40 + verified * 0.35 + completeness * 0.25

    dim4 = recency_log * 0.40 + resp_score * 0.40 + mkt_avail * 0.20

    # ── DIM 5: Logistics ──────────────────────────────────────────────────────
    notice = int(redrob.get("notice_period_days") or 60)
    np_score = 1.00 if notice <= 15 else (0.90 if notice <= 30 else (0.65 if notice <= 60 else (0.40 if notice <= 90 else 0.15)))

    loc_raw  = (profile.get("location") or "").lower()
    country  = (profile.get("country")  or "").lower()
    relocate = bool(redrob.get("willing_to_relocate"))
    loc_score = 0.30
    for city, score in PREFERRED_LOCATIONS.items():
        if city in loc_raw:
            loc_score = score
            break
    else:
        india = "india" in country or country.strip() == "in"
        loc_score = (0.70 if relocate else 0.40) if india else (0.20 if relocate else 0.05)

    wm = (redrob.get("preferred_work_mode") or "flexible").lower()
    wm_score = {"hybrid": 1.0, "flexible": 1.0, "onsite": 0.8, "remote": 0.5}.get(wm, 0.70)

    dim5 = np_score * 0.50 + loc_score * 0.30 + wm_score * 0.20

    # ── Experience Fit Multiplier ─────────────────────────────────────────────
    ef = (1.00 if 6 <= yoe <= 8 else
          0.95 if 5 <= yoe <= 9 else
          0.85 if 4 <= yoe <= 10 else
          0.70 if 3 <= yoe <= 12 else 0.55)

    # ── Hard DQ Multiplier ────────────────────────────────────────────────────
    dq = 1.0
    if career and all(is_services_firm(r.get("company") or "") for r in career):
        dq = 0.05
    elif not any("python" in sn for sn in skill_names):
        dq = 0.10
    elif yoe < 3:
        dq = 0.20

    # ── Honeypot Signals ──────────────────────────────────────────────────────
    yoe_months  = yoe * 12
    hp_signals  = []
    total_dur   = sum(int(r.get("duration_months") or 0) for r in career)
    if yoe_months > 0 and total_dur > yoe_months * 1.30:
        hp_signals.append(0.80)
    expert_zero = sum(1 for s in skills if s.get("proficiency") == "expert" and (s.get("duration_months") or 0) == 0)
    if expert_zero >= 5:
        hp_signals.append(clamp(expert_zero / 10.0))
    hi_prof = sum(1 for s in skills if s.get("proficiency") in ("expert", "advanced"))
    if hi_prof > 15:
        hp_signals.append(0.70)
    perfect = (
        (redrob.get("profile_completeness_score") or 0) == 100,
        (redrob.get("recruiter_response_rate")    or 0) >= 1.0,
        (redrob.get("interview_completion_rate")  or 0) >= 1.0,
        (redrob.get("offer_acceptance_rate")      or 0) >= 1.0,
        (redrob.get("github_activity_score")      or 0) >= 100,
    )
    if sum(perfect) >= 4:
        hp_signals.append(0.85)
    if yoe_months > 0 and any((s.get("duration_months") or 0) > yoe_months for s in skills):
        hp_signals.append(0.90)

    hp_prob  = max(hp_signals) if hp_signals else 0.0
    is_honey = hp_prob > 0.60

    # ── Composite ─────────────────────────────────────────────────────────────
    composite = (
        dim1 * 0.30 + dim2 * 0.25 + dim3 * 0.20 + dim4 * 0.15 + dim5 * 0.10
    )
    composite = clamp(composite * ef * dq)
    if is_honey:
        composite *= 0.05

    # ── Boolean JD flags (useful as LTR binary features) ─────────────────────
    flag_has_python      = any("python" in sn for sn in skill_names)
    flag_has_embeddings  = any(skill_hit(sn, MUST_HAVE_SKILLS["embeddings"]) for sn in skill_names)
    flag_has_vectordb    = any(skill_hit(sn, MUST_HAVE_SKILLS["vector_databases"]) for sn in skill_names)
    flag_has_ranking     = any(skill_hit(sn, MUST_HAVE_SKILLS["ranking_evaluation"]) for sn in skill_names)
    flag_all_must_haves  = all([flag_has_python, flag_has_embeddings, flag_has_vectordb, flag_has_ranking])
    flag_open_to_work    = bool(redrob.get("open_to_work_flag"))
    flag_product_career  = product_ratio >= 0.60
    flag_no_services_dq  = not (career and all(is_services_firm(r.get("company") or "") for r in career))

    return {
        "candidate_id":           c.get("candidate_id"),
        # ── Rubric dimensions ──
        "dim_product_experience": round(dim1, 6),
        "dim_ai_depth":           round(dim2, 6),
        "dim_scrappiness":        round(dim3, 6),
        "dim_availability":       round(dim4, 6),
        "dim_logistics":          round(dim5, 6),
        # ── Multipliers ────────
        "experience_fit":         round(ef, 4),
        "dq_multiplier":          round(dq, 4),
        "honeypot_prob":          round(hp_prob, 4),
        "is_honeypot":            int(is_honey),
        # ── Composite ──────────
        "composite_score":        round(composite, 6),
        "pseudo_label":           -1 if is_honey else round(composite, 6),
        # ── Decay features ─────────────────────────────────────────────────
        "recency_exp_decay":      round(recency_exp, 6),
        "recency_log_decay":      round(recency_log, 6),
        "response_rate_decayed":  round(response_rate_decayed, 6),
        "days_since_active":      round(dsince, 1),
        # ── Raw signals ────────────────────────────────────────────────────
        "yoe":                    yoe,
        "product_ratio":          round(product_ratio, 4),
        "total_career_months":    total_months,
        "product_months":         product_months,
        "prod_evidence_count":    prod_evidence,
        "short_stints_count":     short_stints,
        "must_have_groups_hit":   groups_hit,
        "bonus_groups_hit":       bonus_hit,
        "github_score":           github_raw if github_raw >= 0 else -1,
        "notice_days":            notice,
        "recruiter_response_rate": rr,
        "interview_completion_rate": clamp(redrob.get("interview_completion_rate") or 0.5),
        "profile_completeness":   clamp((redrob.get("profile_completeness_score") or 50) / 100.0),
        # ── Boolean JD flags ──────────────────────────────────────────────
        "flag_has_python":        int(flag_has_python),
        "flag_has_embeddings":    int(flag_has_embeddings),
        "flag_has_vectordb":      int(flag_has_vectordb),
        "flag_has_ranking":       int(flag_has_ranking),
        "flag_all_must_haves":    int(flag_all_must_haves),
        "flag_open_to_work":      int(flag_open_to_work),
        "flag_product_career":    int(flag_product_career),
        "flag_no_services_dq":    int(flag_no_services_dq),
    }


print("Computing tabular features ...")
feature_records = [build_features(c) for c in tqdm(candidates, desc="Features")]

features_df = pd.DataFrame(feature_records)
print(f"\n✅ Features computed: {features_df.shape}")
print(f"   Honeypots flagged: {features_df['is_honeypot'].sum():,}")
print(f"   Score range: [{features_df['composite_score'].min():.4f}, {features_df['composite_score'].max():.4f}]")

FEAT_PATH = ARTIFACTS_DIR / "features.parquet"
features_df.to_parquet(FEAT_PATH, index=False)
print(f"\n💾 Saved → {FEAT_PATH}")

Computing tabular features ...


Features: 100%|██████████| 19607/19607 [00:09<00:00, 2147.19it/s]



✅ Features computed: (19607, 37)
   Honeypots flagged: 3,534
   Score range: [0.0007, 0.7396]

💾 Saved → c:\Home\Projects\Mainstream\Chronos\India_runs_data_and_ai_challenge\features.parquet


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — Embed JD Query & Validate Retrieval Quality
# This is a sanity check — not a submission artifact.
# ─────────────────────────────────────────────────────────────────────────────

print("Encoding JD query ...")
jd_embedding = model.encode(
    [JD_QUERY_TEXT],
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype(np.float32)

# ── FAISS retrieval ──────────────────────────────────────────────────────────
index.nprobe = 20    # search more cells for better recall
distances, faiss_indices = index.search(jd_embedding, 20)

print("\n📊 Top-10 Dense Retrieval Results (FAISS):")
print(f"{'Rank':<6} {'CandidateID':<15} {'Score':<8} {'Preview'}")
print("-" * 80)
for rank, (idx, dist) in enumerate(zip(faiss_indices[0][:10], distances[0][:10]), 1):
    cid     = candidate_ids[idx]
    preview = profile_texts[idx].replace("search_document: ", "")[:80]
    print(f"{rank:<6} {cid:<15} {dist:.4f}   {preview}...")

# ── BM25 retrieval ───────────────────────────────────────────────────────────
jd_tokens = tokenize_for_bm25(JD_QUERY_TEXT)
bm25_scores = bm25.get_scores(jd_tokens)
top_bm25_idx = np.argsort(bm25_scores)[::-1][:10]

print("\n📊 Top-10 Sparse Retrieval Results (BM25):")
print(f"{'Rank':<6} {'CandidateID':<15} {'Score':<8}")
print("-" * 40)
for rank, idx in enumerate(top_bm25_idx, 1):
    cid   = candidate_ids[idx]
    score = bm25_scores[idx]
    print(f"{rank:<6} {cid:<15} {score:.4f}")

# ── Save JD embedding for Rust runtime ───────────────────────────────────────
JD_EMB_PATH = ARTIFACTS_DIR / "jd_query_embedding.npy"
np.save(JD_EMB_PATH, jd_embedding.astype(np.float16))
print(f"\n💾 Saved JD embedding → {JD_EMB_PATH}")



## Cell 12 — Summary Stats & Quality Report


# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — Quality Report (optional, run after all artifacts are saved)
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("PHASE 2 ARTIFACT SUMMARY")
print("=" * 60)

total = len(candidates)
honeypots = features_df["is_honeypot"].sum()
dq_services = (features_df["dq_multiplier"] == 0.05).sum()
dq_no_python = (features_df["dq_multiplier"] == 0.10).sum()

print(f"\n📁 Candidates processed : {total:,}")
print(f"   Honeypots flagged     : {honeypots:,} ({honeypots/total*100:.1f}%)")
print(f"   DQ — services only    : {dq_services:,}")
print(f"   DQ — no Python        : {dq_no_python:,}")

print(f"\n📊 Score Distribution:")
print(features_df["composite_score"].describe().round(4).to_string())

print(f"\n📊 Must-Have Coverage:")
for col in ["flag_has_python", "flag_has_embeddings", "flag_has_vectordb", "flag_has_ranking", "flag_all_must_haves"]:
    pct = features_df[col].mean() * 100
    print(f"   {col:<30} {pct:.1f}%")

print(f"\n📊 Embedding Artifact:")
print(f"   Shape:      {embeddings_f16.shape}")
print(f"   Dtype:      {embeddings_f16.dtype}")
print(f"   Footprint:  {embeddings_f16.nbytes / 1024**2:.1f} MB")
print(f"   FAISS idx:  {FAISS_PATH.stat().st_size / 1024**2:.1f} MB")
print(f"   BM25 idx:   {BM25_PATH.stat().st_size / 1024**2:.1f} MB")
print(f"   Features:   {FEAT_PATH.stat().st_size / 1024**2:.1f} MB")

print("\n✅ Phase 2 complete. Download artifacts and place in Chronos project root.")
print("   Next: Phase 3 — Constrained Rust-Native Execution Sandbox")

Encoding JD query ...

📊 Top-10 Dense Retrieval Results (FAISS):
Rank   CandidateID     Score    Preview
--------------------------------------------------------------------------------
1      CAND_0008425    0.7785   Title: Senior NLP Engineer
Company: Ola
Experience: 7.8 years
Skills: Learning t...
2      CAND_0005260    0.7773   Title: Senior NLP Engineer
Company: Netflix
Experience: 5.2 years
Skills: Semant...
3      CAND_0011687    0.7752   Title: Senior NLP Engineer
Company: Niramai
Experience: 7.8 years
Skills: Tensor...
4      CAND_0046064    0.7750   Title: Senior NLP Engineer
Company: Salesforce
Experience: 8.9 years
Skills: Pyt...
5      CAND_0055905    0.7733   Title: Senior Machine Learning Engineer
Company: Flipkart
Experience: 8.1 years
...
6      CAND_0046525    0.7713   Title: Senior Machine Learning Engineer
Company: Genpact AI
Experience: 6.1 year...
7      CAND_0086022    0.7648   Title: Senior Applied Scientist
Company: Sarvam AI
Experience: 5.3 years
Skills:...
8 

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — Bulletproof Overnight LLM Reasoning Generation (Llama 3.2)
# ─────────────────────────────────────────────────────────────────────────────
import json
import os
import asyncio
import random
import gzip
import pandas as pd
from tqdm.auto import tqdm
import ollama

# ── Config ──
BASE_DIR_NEW = r'c:\Users\VIVEK\Chronos\output'
# Ensure you define the input directory properly (update if your candidates.jsonl is elsewhere)
INPUT_FILE = r'c:\Home\Projects\Mainstream\Chronos\India_runs_data_and_ai_challenge\candidates.jsonl' 

FEATURES_FILE = os.path.join(BASE_DIR_NEW, "features.parquet")
CANDIDATES_FILE = INPUT_FILE
OUTPUT_REASONING_FILE = os.path.join(BASE_DIR_NEW, "precomputed_reasonings.json")
TMP_REASONING_FILE = os.path.join(BASE_DIR_NEW, "precomputed_reasonings.tmp")

OLLAMA_MODEL = "llama3.2"
CHECKPOINT_INTERVAL = 5   # 50 is better for speed. Saving every 5 slows down the event loop.
CONCURRENCY_LIMIT = 2      

async def generate_reasoning(client: ollama.AsyncClient, sem: asyncio.Semaphore, summary: dict) -> tuple[str, str]:
    # variance_instructions = [
    #     "Lead with their specific technical strengths, then state their tenure at their listed companies.",
    #     "Start by naming their most recent employer, then evaluate how their exact years of experience maps to the AI Engineer role.",
    #     "Acknowledge their biggest missing requirement first (e.g., lack of product company experience or core ML skills), then counter with their strongest asset.",
    #     "Begin exactly with their years of experience as the subject of the sentence, followed by a punchy assessment of their fit.",
    #     "Write a direct executive summary focusing on their product-company background (or lack thereof) without using the words 'Although', 'With', or 'This candidate'."
    # ]
    # instruction = random.choice(variance_instructions)
    
    prompt = f"""You are a Principal AI Recruiter writing a final hiring justification for a Senior AI Engineer role. 
The role requires: 5-9 years experience, Production ML, embeddings, Python, and product-company background.

Candidate Data:
- Title: {summary['title']}
- Years of Exp: {summary['yoe']}
- Skills: {summary['skills']}
- Companies: {summary['companies']}

CRITICAL RULES FOR YOUR OUTPUT:
1. Write exactly 1 or 2 sentences.
2. Explicitly state their exact years of experience (e.g., '{summary['yoe']} years').
3. Explicitly name at least one of their companies.
4. Assess their skills against the JD requirements. Acknowledge any gaps honestly.
5. {instruction}
6. Output ONLY the raw justification text. No introductory text, no markdown, no quotation marks.

Justification:"""

    async with sem:
        try:
            response = await client.generate(
                model=OLLAMA_MODEL,
                prompt=prompt,
                options={
                    "temperature": 0.15,
                    "num_predict": 150,
                    "top_p": 0.9,
                }
            )
            # Defensive clean for CSV injection prevention
            reasoning = response['response'].strip()
            reasoning = reasoning.replace('"', '').replace('\n', ' ').replace('\r', '').replace('\t', ' ')
            
            if reasoning.lower().startswith("here is"):
                reasoning = reasoning.split(":", 1)[-1].strip()
            return summary['candidate_id'], reasoning
            
        except Exception as e:
            fallback = f"Brings {summary['yoe']} years of experience as a {summary['title']}, including time at {summary['companies'].split(',')[0]}. Familiar with {summary['skills'].split(',')[0]}, though requires further evaluation against core production ML requirements."
            return summary['candidate_id'], fallback

async def build_reasoning_artifact():
    print("\n[Initialization] Parsing features to target ALL valid candidates...")
    features_df = pd.read_parquet(FEATURES_FILE)
    target_cids = set(features_df["candidate_id"])
    print(f"Targeting {len(target_cids)} total candidates for offline reasoning generation.")
    
    reasoning_dict = {}
    if os.path.exists(OUTPUT_REASONING_FILE):
        try:
            with open(OUTPUT_REASONING_FILE, "r", encoding="utf-8") as f:
                reasoning_dict = json.load(f)
            print(f"[Checkpoint] Resuming from existing artifact. {len(reasoning_dict)} candidates already processed.")
        except json.JSONDecodeError:
            print("[Warning] Corrupted JSON found. Starting fresh.")
    
    targets_remaining = target_cids - set(reasoning_dict.keys())
    
    if not targets_remaining:
        print("[Status] All target candidates have precomputed reasoning. Exiting.")
        return

    print(f"[Data Extraction] Building prompt payloads for {len(targets_remaining)} remaining candidates...")
    summaries = []
    
    opener = gzip.open if str(CANDIDATES_FILE).endswith(".gz") else open
    with opener(CANDIDATES_FILE, "rt", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            c = json.loads(line)
            cid = c.get("candidate_id")
            
            if cid in targets_remaining:
                prof = c.get("profile", {})
                career = c.get("career_history", [])
                skills = [s.get("name") for s in c.get("skills", [])[:4]]
                companies = [r.get("company") for r in career[:2]]
                
                summaries.append({
                    "candidate_id": cid,
                    "title": prof.get("current_title", "Engineer"),
                    "yoe": prof.get("years_of_experience", 0),
                    "skills": ", ".join([str(s) for s in skills if s]),
                    "companies": ", ".join([str(co) for co in companies if co])
                })
                if len(summaries) >= len(targets_remaining):
                    break

    print(f"\n[Inference] Executing Llama 3.2 generation at ~79 t/s...")
    client = ollama.AsyncClient()
    sem = asyncio.Semaphore(CONCURRENCY_LIMIT)
    
    pending_tasks = [generate_reasoning(client, sem, s) for s in summaries]
    
    try:
        for i in tqdm(range(0, len(pending_tasks), CHECKPOINT_INTERVAL), desc="Batch Processing"):
            chunk = pending_tasks[i : i + CHECKPOINT_INTERVAL]
            chunk_results = await asyncio.gather(*chunk)
            
            for cid, text in chunk_results:
                reasoning_dict[cid] = text
                
            # Atomic Save: Write to Temp, then Rename
            with open(TMP_REASONING_FILE, "w", encoding="utf-8") as f:
                json.dump(reasoning_dict, f, indent=2)
            os.replace(TMP_REASONING_FILE, OUTPUT_REASONING_FILE)
                
    except KeyboardInterrupt:
        print("\n[Halted] User interrupt detected. Finalizing checkpoint...")
    finally:
        with open(TMP_REASONING_FILE, "w", encoding="utf-8") as f:
            json.dump(reasoning_dict, f, indent=2)
        os.replace(TMP_REASONING_FILE, OUTPUT_REASONING_FILE)
        
        filename = os.path.basename(OUTPUT_REASONING_FILE)
        print(f"\n[Complete] 💾 Saved {len(reasoning_dict)} Precomputed Reasonings → {filename}")

# Execute directly in the existing Jupyter/Colab event loop
await build_reasoning_artifact()


[Initialization] Parsing features to target ALL valid candidates...
Targeting 19933 total candidates for offline reasoning generation.
[Warning] Corrupted JSON found. Starting fresh.
[Data Extraction] Building prompt payloads for 19933 remaining candidates...

[Inference] Executing Llama 3.2 generation at ~79 t/s...


Batch Processing: 100%|██████████| 3987/3987 [4:36:16<00:00,  4.16s/it]  


[Complete] 💾 Saved 19933 Precomputed Reasonings → precomputed_reasonings.json
